In [3]:
# CELDA 20: CONFIGURAR FASTAPI CON ENDPOINT /predict

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
import joblib
import pandas as pd
import numpy as np
import os
import threading
import uvicorn

# Crear instancia de FastAPI
app = FastAPI(
    title="RAWG Games API",
    description="API para predecir éxito de juegos y responder preguntas sobre videojuegos",
    version="1.0",
    contact={
        "name": "Manuel Serrano",
        "email": "manuel.serrano@example.com"
    }
)

# Verificar que el modelo existe
if not os.path.exists("models/model.pkl"):
    raise FileNotFoundError("❌ Modelo no encontrado: models/model.pkl\nEjecuta primero 04_modelo_ml.ipynb (Celdas 16-19)")

# Cargar modelo y features
print("🔄 Cargando modelo entrenado...")
model = joblib.load("models/model.pkl")
feature_names = joblib.load("models/feature_names.pkl")
print(f"✅ Modelo cargado correctamente")
print(f"   - Features: {len(feature_names)}")
print(f"   - Tipo de modelo: {type(model).__name__}")

# Definir esquema de entrada para /predict
class GameInput(BaseModel):
    """Estructura de entrada para predecir éxito de un juego"""
    release_year: int
    rating: float
    ratings_count: int
    metacritic: int
    playtime: int
    status_yet: int
    status_owned: int
    status_beaten: int
    status_toplay: int
    status_dropped: int
    status_playing: int
    esrb_rating: str = "Unknown"
    genres_list: str = "Unknown"
    platforms_list: str = "Unknown"

@app.post("/predict", summary="Predice si un juego será exitoso")
def predict_success(game: GameInput):
    """
    Predice si un juego será exitoso basado en sus características.
    
    - Usa el modelo XGBoost entrenado con datos históricos de RAWG
    - Devuelve probabilidad de éxito y clasificación binaria
    """
    try:
        # Convertir entrada a DataFrame
        input_data = pd.DataFrame([game.dict()])
        
        # Asegurar que tiene todas las features en el orden correcto
        input_data = input_data[feature_names]
        
        # Preprocesar categóricos (mismo que en entrenamiento)
        for col in ["esrb_rating", "genres_list", "platforms_list"]:
            if col in input_data.columns:
                input_data[col] = input_data[col].fillna("Unknown").astype(str)
        
        # Predecir
        prediction = model.predict(input_data)[0]
        probability = model.predict_proba(input_data)[0][1]
        
        return {
            "success": bool(prediction),
            "probability": round(float(probability), 4),
            "message": "✅ Juego exitoso" if prediction else "❌ Juego no exitoso",
            "features_used": len(feature_names)
        }
        
    except Exception as e:
        raise HTTPException(
            status_code=400,
            detail=f"Error al procesar la predicción: {str(e)}"
        )

@app.get("/health", summary="Verifica estado de la API")
def health_check():
    """Endpoint para verificar que la API está funcionando"""
    return {
        "status": "ok",
        "model_loaded": True,
        "features_count": len(feature_names),
        "timestamp": pd.Timestamp.now().isoformat()
    }

print("\n✅ FastAPI configurado correctamente")
print("   - Endpoint /predict listo para recibir predicciones")
print("   - Endpoint /health para monitoreo")
print("\n🚀 Para ejecutar el servidor:")
print("   uvicorn 05_api_endpoints:app --reload --port 8000")
print("\n🌐 Acceder a la documentación interactiva:")
print("   http://localhost:8000/docs")

🔄 Cargando modelo entrenado...
✅ Modelo cargado correctamente
   - Features: 14
   - Tipo de modelo: XGBClassifier

✅ FastAPI configurado correctamente
   - Endpoint /predict listo para recibir predicciones
   - Endpoint /health para monitoreo

🚀 Para ejecutar el servidor:
   uvicorn 05_api_endpoints:app --reload --port 8000

🌐 Acceder a la documentación interactiva:
   http://localhost:8000/docs


In [4]:
# CELDA 21: EJECUTAR SERVIDOR FASTAPI EN SEGUNDO PLANO
# Fuente: archivo 16.FastAPI.ipynb

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

# Iniciar servidor en hilo separado
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

print("🚀 Servidor FastAPI iniciado en segundo plano")
print("   - URL base: http://localhost:8000")
print("   - Documentación: http://localhost:8000/docs")
print("   - Health check: http://localhost:8000/health")

🚀 Servidor FastAPI iniciado en segundo plano
   - URL base: http://localhost:8000
   - Documentación: http://localhost:8000/docs
   - Health check: http://localhost:8000/health


INFO:     Started server process [5616]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:61848 - "GET /docs HTTP/1.1" 200 OK


email-validator not installed, email fields will be treated as str.
To install, run: pip install email-validator


INFO:     127.0.0.1:61848 - "GET /openapi.json HTTP/1.1" 200 OK
INFO:     127.0.0.1:64528 - "GET /docs HTTP/1.1" 200 OK
INFO:     127.0.0.1:64528 - "GET /openapi.json HTTP/1.1" 200 OK
